## 1. Understanding torch.manual_seed()
What is Random Seed?
A random seed initializes the random number generator to produce the same sequence of "random" numbers every time. This is crucial for reproducibility in machine learning experiments.

In [3]:
import torch

# Without setting seed - different results each time
print("Without seed:")
print("Run 1:", torch.rand(3))
print("Run 2:", torch.rand(3))

# With seed - same results each time
print("\nWith seed:")
torch.manual_seed(1337)
print("Run 1:", torch.rand(3))

torch.manual_seed(1337)
print("Run 2:", torch.rand(3))  # Same as Run 1!

Without seed:
Run 1: tensor([0.8727, 0.9077, 0.0024])
Run 2: tensor([0.9386, 0.0432, 0.1551])

With seed:
Run 1: tensor([0.0783, 0.4956, 0.6231])
Run 2: tensor([0.0783, 0.4956, 0.6231])


## 2. Loss Functions
Loss functions measure how far predictions are from true values. The goal is to minimize this loss.

### 2.1 MSELoss (Mean Squared Error)
Use Case: Regression problems (predicting continuous values)
Formula: MSE = (1/n) × Σ(predicted - actual)²

In [4]:
import torch
import torch.nn as nn

# Create loss function
mse_loss = nn.MSELoss()

# True house prices (in thousands)
actual_prices = torch.tensor([300.0, 450.0, 200.0, 550.0])

# Model's predictions
predicted_prices = torch.tensor([290.0, 460.0, 210.0, 530.0])

# Calculate loss
loss = mse_loss(predicted_prices, actual_prices)
print(f"MSE Loss: {loss.item():.2f}")
print(f"Average prediction error: ${loss.item()**0.5:.2f}k")

MSE Loss: 175.00
Average prediction error: $13.23k


In [11]:
torch.manual_seed(42)

# Simple dataset: predicting y = 2x + 1
X = torch.randn(100, 1) * 10  # 100 samples
y = 2 * X + 1 + torch.randn(100, 1) * 2  # Add noise

# Simple linear model
model = nn.Linear(1, 1)
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(100):
    # Forward pass
    predictions = model(X)
    loss = mse_loss(predictions, y)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print(f"\nLearned parameters: weight={model.weight.item():.2f}, bias={model.bias.item():.2f}")
print(f"True parameters: weight=2.00, bias=1.00")

Epoch 0, Loss: 227.6187
Epoch 20, Loss: 172.6350
Epoch 40, Loss: 127.3854
Epoch 60, Loss: 91.8290
Epoch 80, Loss: 64.8029

Learned parameters: weight=1.34, bias=1.47
True parameters: weight=2.00, bias=1.00


### 2.2 BCELoss (Binary Cross-Entropy Loss)
Use Case: Binary classification (2 classes: 0 or 1)

Formula: BCE = -1/n × Σ[y×log(p) + (1-y)×log(1-p)]

Where:

y = true label (0 or 1)
p = predicted probability

In [12]:
import torch
import torch.nn as nn

# Binary Cross-Entropy Loss
bce_loss = nn.BCELoss()

# True labels: 1 = spam, 0 = not spam
actual = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])

# Model's predicted probabilities (must be between 0 and 1)
predicted = torch.tensor([0.9, 0.1, 0.8, 0.3, 0.95])

loss = bce_loss(predicted, actual)
print(f"BCE Loss: {loss.item():.4f}")

BCE Loss: 0.1684


In [16]:
torch.manual_seed(1337)

# Dataset: classify points above/below a line
X = torch.randn(200, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

# Neural network
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 10)
        self.layer2 = nn.Linear(10, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model = BinaryClassifier()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(100):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        accuracy = ((predictions > 0.5) == y).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.2%}")

Epoch 0, Loss: 0.8217, Accuracy: 42.00%
Epoch 20, Loss: 0.5219, Accuracy: 83.50%
Epoch 40, Loss: 0.2971, Accuracy: 93.00%
Epoch 60, Loss: 0.1656, Accuracy: 96.50%
Epoch 80, Loss: 0.1070, Accuracy: 97.00%


## 3. Optimizers
Optimizers adjust model weights to minimize loss. They control how the model learns.
### 3.1 SGD (Stochastic Gradient Descent)
How it works: Updates weights in the direction that reduces loss, scaled by learning rate.

Update Rule: weight = weight - learning_rate × gradient

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

# Simple model
model = nn.Linear(10, 1)

# SGD optimizer
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Dummy data
X = torch.randn(32, 10)
y = torch.randn(32, 1)

criterion = nn.MSELoss()

# Training step
predictions = model(X)
loss = criterion(predictions, y)

# Update weights
optimizer.zero_grad()  # Clear previous gradients
loss.backward()         # Compute gradients
optimizer.step()        # Update weights

print(f"Loss: {loss.item():.4f}")

Loss: 0.9897


### 3.2 Adam (Adaptive Moment Estimation)
How it works: Automatically adjusts learning rate for each parameter based on gradient history. Combines momentum and adaptive learning rates.

Key Features:

    Remembers past gradients (momentum)
    Adapts learning rate per parameter
    Usually faster than SGD
    Default choice for most problems

In [18]:
torch.manual_seed(42)

# Same model
model = nn.Linear(10, 1)

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training step (same as SGD)
X = torch.randn(32, 10)
y = torch.randn(32, 1)
criterion = nn.MSELoss()

predictions = model(X)
loss = criterion(predictions, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Loss: {loss.item():.4f}")

Loss: 0.9897


In [19]:
torch.manual_seed(100)

# Dataset
X = torch.randn(1000, 20)
y = torch.randn(1000, 1)

# Function to train with different optimizers
def train_model(optimizer_name, lr=0.01, epochs=50):
    model = nn.Sequential(
        nn.Linear(20, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )
    
    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)
    
    criterion = nn.MSELoss()
    losses = []
    
    for epoch in range(epochs):
        predictions = model(X)
        loss = criterion(predictions, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
    
    return losses

# Train with both optimizers
sgd_losses = train_model("SGD", lr=0.01)
adam_losses = train_model("Adam", lr=0.01)

print(f"SGD final loss: {sgd_losses[-1]:.4f}")
print(f"Adam final loss: {adam_losses[-1]:.4f}")

SGD final loss: 0.9852
Adam final loss: 0.4199


## 4. Complete End-to-End Example
Problem: Predict Student Test Scores

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(1337)

# Generate synthetic data
# Features: [study_hours, previous_score, attendance_rate]
num_samples = 500
study_hours = torch.randn(num_samples, 1) * 5 + 10
previous_score = torch.randn(num_samples, 1) * 10 + 70
attendance = torch.rand(num_samples, 1) * 30 + 70

X = torch.cat([study_hours, previous_score, attendance], dim=1)

# True relationship: score = 2*study + 0.5*previous + 1*attendance + noise
y = (2 * study_hours + 0.5 * previous_score + 1 * attendance + 
     torch.randn(num_samples, 1) * 5)

# Split into train/test
train_size = 400
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Neural Network Model
class ScorePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = ScorePredictor()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
print("Training Score Predictor...")
for epoch in range(200):
    # Forward pass
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 40 == 0:
        # Evaluate on test set
        with torch.no_grad():
            test_pred = model(X_test)
            test_loss = criterion(test_pred, y_test)
        
        print(f"Epoch {epoch:3d} | Train Loss: {loss.item():6.2f} | Test Loss: {test_loss.item():6.2f}")

# Final evaluation
print("\n" + "="*50)
print("Final Model Performance:")
print("="*50)

with torch.no_grad():
    train_pred = model(X_train)
    test_pred = model(X_test)
    
    train_mse = criterion(train_pred, y_train)
    test_mse = criterion(test_pred, y_test)
    
    print(f"Train MSE: {train_mse.item():.2f}")
    print(f"Test MSE:  {test_mse.item():.2f}")
    print(f"Average prediction error: ±{test_mse.item()**0.5:.2f} points")

# Example predictions
print("\n" + "="*50)
print("Sample Predictions:")
print("="*50)
print(f"{'Study Hrs':<10} {'Prev Score':<12} {'Attend %':<10} {'Predicted':<12} {'Actual'}")
print("-" * 60)

for i in range(5):
    study, prev, attend = X_test[i]
    pred = test_pred[i].item()
    actual = y_test[i].item()
    print(f"{study.item():<10.1f} {prev.item():<12.1f} {attend.item():<10.1f} {pred:<12.1f} {actual:.1f}")

Training Score Predictor...
Epoch   0 | Train Loss: 20382.41 | Test Loss: 18890.01
Epoch  40 | Train Loss: 159.53 | Test Loss: 156.15
Epoch  80 | Train Loss:  80.09 | Test Loss:  69.21
Epoch 120 | Train Loss:  63.65 | Test Loss:  62.86
Epoch 160 | Train Loss:  52.85 | Test Loss:  51.18

Final Model Performance:
Train MSE: 43.90
Test MSE:  42.58
Average prediction error: ±6.53 points

Sample Predictions:
Study Hrs  Prev Score   Attend %   Predicted    Actual
------------------------------------------------------------
10.5       71.7         93.8       149.5        153.9
10.3       84.3         75.6       142.7        138.3
4.3        65.0         82.8       126.9        125.7
4.8        64.7         89.4       133.2        131.5
8.7        72.2         94.6       148.3        149.0
